# AI in your Automation Stack — webhook-shaped

Three webhook-shaped payloads (HubSpot lead, support ticket, Slack message) → one LLM enrichment function → enriched JSON output ready to send back into your automation.

This is what's happening *under the hood* when you drop an Eden AI step into n8n / Zapier / Make. The cookbook shows the pattern in plain Python so you can fork it into your own backend before going no-code (or build your own n8n-equivalent in-house).

**Prerequisites:** an Eden AI API key (set as `EDENAI_API_KEY` env var or in `.env`).

In [ ]:
%pip install --quiet requests ipywidgets python-dotenv

## 1. Configuration

In [ ]:
import base64
import json
import os

from dotenv import load_dotenv
from IPython.display import HTML, display

load_dotenv(override=True)

EDENAI_API_KEY = os.environ.get("EDENAI_API_KEY")
if not EDENAI_API_KEY:
    raise RuntimeError("Set EDENAI_API_KEY (env var or .env file). Get one at https://app.edenai.run")
EDENAI_URL = "https://api.edenai.run/v3/llm/chat/completions"

LLM_MODEL = "anthropic/claude-sonnet-4-5"

# Three realistic webhook payloads (the kind your CRM / helpdesk / Slack would send)
SAMPLE_PAYLOADS = {
    "HubSpot — new lead": {
        "event":     "contact.created",
        "contact": {
            "email":   "alex.martin@acmecorp.com",
            "name":    "Alex Martin",
            "company": "Acme Corp",
            "phone":   "+1 415 555 0117",
        },
        "form": {
            "page":    "/pricing",
            "message": "We're a 50-person series B startup looking for SOC2-compliant observability for our ML pipelines. Need pricing for the Enterprise tier and timeline to onboard 3 teams. Urgent — board meeting next week.",
        },
    },
    "Support — new ticket": {
        "event":   "ticket.created",
        "ticket": {
            "id":      "T-8847",
            "author":  "jane.smith@bloombottle.com",
            "channel": "email",
            "subject": "Drift detection broken since Tuesday",
            "body":    "Hey team, since around 2pm Tuesday our drift alerts stopped firing entirely. We're on the Team plan. Production pipeline still ingesting events normally per the dashboard. Already tried restarting our SDK — same. This is blocking our Q1 launch readiness review. Please prioritize.",
        },
    },
    "Slack — community message": {
        "event":   "message.posted",
        "channel": "#community-help",
        "user":    "@maria",
        "text":    "is anyone else seeing weird latency spikes in the EU region today? my p99 went from 80ms to 450ms around 11 UTC, no changes on my side. is this a cinder thing or am I crazy 🙃",
    },
}


def _is_sandbox(jwt: str) -> bool:
    try:
        payload_b64 = jwt.split(".")[1]
        payload_b64 += "=" * (-len(payload_b64) % 4)
        return json.loads(base64.urlsafe_b64decode(payload_b64)).get("type") == "sandbox_api_token"
    except Exception:
        return False


if _is_sandbox(EDENAI_API_KEY):
    display(HTML(
        '<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px 14px;'
        'border-radius:4px;font-family:sans-serif;font-size:13px;margin:6px 0;">'
        '<b>⚠ Sandbox key detected.</b> Use a production key for real enrichment.</div>'
    ))

## 2. The enrichment function

A single function: takes any webhook-shaped payload, asks the LLM to enrich it with derived fields (intent, sentiment, urgency, entities, suggested action), returns the original payload with an `_ai_enrichment` block appended. This is exactly what an `n8n` AI node / Zapier path / Make scenario would call.

In [ ]:
import time

import requests

MAX_RETRIES = 2

ENRICHMENT_PROMPT = """You are an AI enrichment step in an automation pipeline. Given the webhook payload below, return ONLY a JSON object (no prose, no fences) with these keys:

- intent:          one of ["sales_inquiry", "support_request", "bug_report", "feedback", "question", "complaint", "other"]
- urgency:         one of ["low", "medium", "high", "critical"]
- sentiment:       one of ["positive", "neutral", "negative"]
- entities:        array of {type, value} — extract product names, plan tiers, ticket IDs, error codes, deadlines, dollar amounts
- summary:         one-sentence summary of what the sender wants
- suggested_route: one of ["sales_team", "support_l1", "support_l2", "engineering_oncall", "community_mod", "autorespond", "ignore"]
- suggested_reply: one short sentence the system could auto-send as an immediate acknowledgment (or null if the route is "engineering_oncall" or "ignore")

Rules:
- Be conservative with "critical" urgency — only when there's a deadline within 24h or production impact.
- Entities must be present in the payload (no hallucinated dollar amounts or deadlines).

PAYLOAD:
"""


def _strip_fences(text):
    t = text.strip()
    if t.startswith("```"):
        t = t.split("\n", 1)[1] if "\n" in t else t[3:]
        if t.endswith("```"):
            t = t.rsplit("```", 1)[0]
    return t.strip()


def enrich(payload: dict) -> dict:
    """Send a webhook payload to the LLM, return payload with `_ai_enrichment` appended.

    This is exactly the shape of an n8n / Zapier / Make AI step:
    input dict → enriched dict, side-effect free. Drop into a Flask route
    or a Lambda and you're done.
    """
    headers = {"Authorization": f"Bearer {EDENAI_API_KEY}", "Content-Type": "application/json"}
    body = {
        "model":    LLM_MODEL,
        "messages": [{"role": "user", "content": ENRICHMENT_PROMPT + json.dumps(payload, indent=2)}],
    }
    start = time.perf_counter()
    last_err = None
    for attempt in range(MAX_RETRIES + 1):
        r = requests.post(EDENAI_URL, headers=headers, json=body, timeout=60)
        if r.status_code == 200:
            content = r.json()["choices"][0]["message"]["content"]
            try:
                enrichment = json.loads(_strip_fences(content))
            except json.JSONDecodeError as e:
                return {**payload, "_ai_enrichment": {"error": f"invalid JSON: {e}", "raw": content[:300]}}
            return {**payload, "_ai_enrichment": {**enrichment, "_latency_ms": int((time.perf_counter() - start) * 1000)}}
        last_err = f"HTTP {r.status_code}: {r.text[:200]}"
        if r.status_code in (400, 429, 502, 503, 504) and attempt < MAX_RETRIES:
            time.sleep(0.6 * (attempt + 1))
            continue
        break
    return {**payload, "_ai_enrichment": {"error": last_err}}

## 3. UI — input + enriched output side-by-side

In [ ]:
from ipywidgets import Button, Dropdown, HBox, HTML as HTMLWidget, Layout, Output, Textarea, VBox
from IPython.display import display, clear_output

sample_dropdown = Dropdown(
    options=list(SAMPLE_PAYLOADS.keys()),
    value="HubSpot — new lead",
    description="Webhook:",
    layout=Layout(width="360px"),
)

input_box = Textarea(
    value=json.dumps(SAMPLE_PAYLOADS[sample_dropdown.value], indent=2),
    layout=Layout(width="100%", height="260px"),
)


def _on_sample_change(change):
    if change["name"] == "value":
        input_box.value = json.dumps(SAMPLE_PAYLOADS[change["new"]], indent=2)


sample_dropdown.observe(_on_sample_change, names="value")

enrich_btn = Button(description="⚙ Enrich", button_style="primary")
clear_btn = Button(description="Clear")

input_label = HTMLWidget(value='<b style="font-family:sans-serif;font-size:13px;color:#666;">📥 INPUT PAYLOAD (your webhook receives this)</b>')
output_label = HTMLWidget(value='<b style="font-family:sans-serif;font-size:13px;color:#666;">📤 ENRICHED PAYLOAD (forwarded to the next step)</b>')

output_view = Output(layout=Layout(border="1px solid #ddd", padding="10px", min_height="260px", overflow="auto"))


def _empty_output():
    output_view.clear_output()
    with output_view:
        display(HTML(
            '<div style="font-family:sans-serif;color:#aaa;font-size:12px;text-align:center;padding:80px 10px;">'
            'Pick a sample payload and click<br><b>⚙ Enrich</b></div>'
        ))


_empty_output()

display(VBox([
    sample_dropdown,
    input_label, input_box,
    HBox([enrich_btn, clear_btn]),
    output_label, output_view,
]))

## 4. Wire it up

In [ ]:
import html as _html

URGENCY_COLOR = {
    "low":      "#28a745",
    "medium":   "#17a2b8",
    "high":     "#fd7e14",
    "critical": "#dc3545",
}
SENTIMENT_COLOR = {"positive": "#28a745", "neutral": "#6c757d", "negative": "#dc3545"}


def _badge(text, color):
    return (
        f'<span style="background:{color};color:white;padding:2px 10px;border-radius:10px;'
        f'font-size:11px;font-weight:600;font-family:sans-serif;margin-right:4px;">{text}</span>'
    )


def _render_enrichment(enriched):
    e = enriched.get("_ai_enrichment") or {}
    if "error" in e:
        output_view.clear_output()
        with output_view:
            display(HTML(
                f'<div style="color:#dc3545;font-family:monospace;font-size:11px;'
                f'padding:8px;background:#f8d7da;border-radius:4px;">'
                f'{_html.escape(str(e.get("error")))[:400]}</div>'
            ))
        return
    badges = (
        _badge(f'intent: {e.get("intent", "?")}', "#007bff") +
        _badge(f'urgency: {e.get("urgency", "?")}', URGENCY_COLOR.get(e.get("urgency"), "#6c757d")) +
        _badge(f'sentiment: {e.get("sentiment", "?")}', SENTIMENT_COLOR.get(e.get("sentiment"), "#6c757d")) +
        _badge(f'→ {e.get("suggested_route", "?")}', "#6610f2")
    )
    entities = e.get("entities") or []
    ent_html = ""
    if entities:
        rows = "".join(
            f'<tr><td style="padding:2px 8px;font-family:monospace;font-size:10px;color:#666;">{_html.escape(str(x.get("type", "")))}</td>'
            f'<td style="padding:2px 8px;font-family:monospace;font-size:10px;">{_html.escape(str(x.get("value", "")))}</td></tr>'
            for x in entities
        )
        ent_html = (
            f'<div style="font-family:sans-serif;font-size:11px;color:#666;margin-top:8px;"><b>Entities</b></div>'
            f'<table style="border-collapse:collapse;width:100%;">{rows}</table>'
        )
    summary = _html.escape(e.get("summary", ""))
    reply = e.get("suggested_reply")
    reply_html = ""
    if reply:
        reply_html = (
            f'<div style="font-family:sans-serif;font-size:11px;color:#666;margin-top:8px;"><b>Suggested auto-reply</b></div>'
            f'<div style="font-family:sans-serif;font-size:12px;background:#e7f3ff;padding:6px 10px;'
            f'border-radius:3px;border-left:3px solid #007bff;">"{_html.escape(reply)}"</div>'
        )

    json_text = json.dumps(enriched, indent=2, ensure_ascii=False)
    safe_json = json_text.replace("\\", "\\\\").replace("`", "\\`").replace("</", "<\\/")
    copy_btn = (
        f'<button onclick="navigator.clipboard.writeText(`{safe_json}`);'
        f'this.textContent=&quot;Copied ✓&quot;;setTimeout(()=>this.textContent=&quot;Copy enriched JSON&quot;,1500);" '
        f'style="font-size:11px;padding:2px 8px;border:1px solid #ddd;background:#fff;'
        f'border-radius:3px;cursor:pointer;float:right;">Copy enriched JSON</button>'
    )

    latency = e.get("_latency_ms")
    latency_html = f'<span style="color:#888;font-size:11px;float:right;margin-right:8px;">{latency} ms</span>' if latency is not None else ""

    output_view.clear_output()
    with output_view:
        display(HTML(
            copy_btn + latency_html +
            f'<div style="clear:both;font-family:sans-serif;font-size:12px;margin-bottom:6px;"><b>Summary:</b> {summary}</div>'
            f'<div style="margin:6px 0;">{badges}</div>'
            f'{ent_html}'
            f'{reply_html}'
            f'<details style="margin-top:12px;"><summary style="cursor:pointer;font-family:sans-serif;font-size:11px;color:#666;">'
            f'Full enriched payload (the JSON your next pipeline step receives)</summary>'
            f'<pre style="font-size:10px;line-height:1.4;background:#f8f9fa;padding:8px;'
            f'border-radius:4px;overflow:auto;margin-top:6px;">{_html.escape(json_text)}</pre></details>'
        ))


def on_enrich(_):
    try:
        payload = json.loads(input_box.value)
    except json.JSONDecodeError as e:
        output_view.clear_output()
        with output_view:
            display(HTML(
                f'<div style="color:#dc3545;font-family:monospace;font-size:11px;padding:8px;'
                f'background:#f8d7da;border-radius:4px;">Input is not valid JSON: {e}</div>'
            ))
        return
    output_view.clear_output()
    with output_view:
        display(HTML(
            '<div style="font-family:sans-serif;color:#17a2b8;font-size:13px;text-align:center;padding:80px 10px;">'
            'enriching… (LLM call ~1–3s)</div>'
        ))
    result = enrich(payload)
    _render_enrichment(result)


def on_clear(_):
    _empty_output()


enrich_btn.on_click(on_enrich)
clear_btn.on_click(on_clear)

## 5. Drop this into a real automation

The `enrich(payload) → enriched_payload` function is side-effect-free and self-contained. Three ways to deploy it:

### As a Flask webhook (~10 lines)
```python
from flask import Flask, request, jsonify
app = Flask(__name__)

@app.post("/enrich")
def webhook():
    return jsonify(enrich(request.get_json()))
```
Point your HubSpot / Zendesk / Slack webhook at this URL — done.

### As an n8n Code node
n8n's Code node runs JavaScript by default, but you can use the `Python (Beta)` node or expose `enrich` over HTTP and call it from an HTTP Request node.

### As a Make.com / Zapier action
Wrap the function in a tiny HTTPS endpoint (Vercel / AWS Lambda / Cloudflare Worker). Make / Zapier hit it as a webhook. The shape of the request and response is intentionally generic so it slots into any "AI Step" in any automation tool.

## 6. Customize

**Different enrichment schema.** Edit `ENRICHMENT_PROMPT` — add fields like `language`, `requires_human_review`, `gdpr_pii_present`, `next_action_due_at`. The downstream pipeline gets richer payloads, the human-in-the-loop step disappears.

**Multiple models, vote.** Run the same payload through 3 models and pick the majority-vote intent/urgency. Reduces variance for high-stakes routing decisions like "is this a P1 or P2 ticket".

**Server-side strict JSON.** Add `response_format: {type: "json_schema", strict: true}` to the LLM call to guarantee well-formed enrichment payloads. See the `document_to_json.ipynb` recipe for the schema pattern.

**Cache enrichments.** Hash the payload, cache the enrichment for ~5 minutes — protects you from duplicate webhook deliveries (every CRM does this) and cuts your LLM bill by 30-70% on noisy event streams.